# 11 — KKT conditions, curvature, and scaling

**Objectives.** Compute first-order optimality residuals, interpret multipliers, and recognize poor scaling. For equalities $g(x)=0$, the Lagrangian is $L=f+\lambda^Tg$ and KKT stationarity is $\nabla f+J_g^T\lambda=0$. Inequalities add sign and complementarity conditions. These are necessary, not generally sufficient, for a local minimum.


In [ ]:
import casadi as ca
import numpy as np
from casadi_tutorial.diagnostics import kkt_residuals
x = ca.SX.sym('x', 2); lam = ca.SX.sym('lam')
f = (x[0]-1)**2 + (x[1]-2)**2; g = x[0]+x[1]-2
grad_f = ca.Function('grad_f',[x],[ca.gradient(f,x)])
jac_g = ca.Function('jac_g',[x],[ca.jacobian(g,x)])
# Analytic constrained solution: x=[0.5,1.5], lambda=1
residual = kkt_residuals(grad_f([0.5,1.5]), jac_g([0.5,1.5]), [1.0], [0.0], [0.0], [0.0])
print(residual); assert residual['stationarity_inf'] < 1e-12


## Scaling

If one decision is near $10^{-6}$ and another near $10^6$, a single solver tolerance has incompatible physical meanings. Introduce dimensionless variables $\hat x=(x-x_{ref})/s_x$, scale residuals by their meaningful magnitudes, and keep objective terms comparable. Scaling changes conditioning, not the physical solution.


In [ ]:
H_bad = np.diag([1e-12, 1e12]); H_scaled = np.eye(2)
print('condition numbers:', np.linalg.cond(H_bad), np.linalg.cond(H_scaled))
assert np.linalg.cond(H_scaled) == 1.0


**Common mistake:** increasing `max_iter` cannot repair inconsistent units or a singular constraint Jacobian. **Recap:** inspect primal feasibility, stationarity, complementarity, curvature, and scaling separately.
